In [1]:
import copy
import os
import sys

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.models.machines import ieee_machine2_params, ieee_machine2, NeuralFlux, _build_cross_coupling
from current_setpoints.models.machines import PMSMDrive
from current_setpoints.utils import (
    NeuralFluxPredictor,
    load_aggregated_csv_data,
    load_neural_flux_model,
)

torch.manual_seed(42)
np.random.seed(42)

In [2]:
AGGREGATED_FILE_PATH = "../data/aggregated_file_means.csv"
COLUMN_MAP = {c: c for c in ["omega", "id1", "iq1", "id3", "iq3", "ud1", "uq1", "ud3", "uq3", "torq"]}
INPUT_SIZE = 5
OUTPUT_SIZE = 1  # scalar co-energy residual -- NOT the flux vector (see notebook overview)
TEST_SIZE = 0.15
RANDOM_STATE = 42
VAL_SIZE = 0.20

# The dataset is tiny (174 rows, full-batch training); CPU is faster here
# than GPU due to per-kernel-launch overhead dominating at this scale.
DEVICE = torch.device("cpu")

ACTIVATION = "gelu"
LR = 5e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 15000
PATIENCE = 1000

# HIDDEN_SIZES/VOLT_WEIGHT: HIDDEN_SIZES=[12] (one hidden layer, size 12)
# chosen from an actual grid sweep (hidden_size in {12,24,32,48} x
# volt_weight in {0.5,1,2,3,4,6,8,10}, gelu, this same seed=42/split), not
# hand-picked. Bigger hidden sizes never dominated H=12 anywhere on the
# sweep's own Pareto frontier -- capacity was not the bottleneck. Best
# joint (volt_rmse + torq_rmse) point: H=12, VW=6 (v=0.60V, t=0.52Nm).
# Voltage is a hard optimizer CONSTRAINT while torque is merely the
# objective, so erring toward voltage accuracy is intentional -- raise/
# lower VOLT_WEIGHT and retrain if that priority should shift.
# NeuralFluxPredictor/NeuralFlux support any number of hidden layers now
# (see machines.py's NeuralFlux._forward_grad_hess) -- e.g. [24, 12] for a
# 2-hidden-layer net, needed for denser/less noisy flux data than this
# measured set (see coordination/inbox.md 2026-07-28).
HIDDEN_SIZES = [12]
VOLT_WEIGHT = 6.0

MODEL_SAVE_PATH = "../weights/FluxNN_Weights.pth"
SCALER_SAVE_PATH = "../weights/FluxNN_Scaler.npy"

params = ieee_machine2_params()
R_stat = torch.from_numpy(params.R_stat).float().to(DEVICE)
L_stat = torch.from_numpy(params.L_stat).float().to(DEVICE)
flux_pm = torch.from_numpy(params.flux_pm).float().to(DEVICE)
J = torch.from_numpy(_build_cross_coupling(2)).float().to(DEVICE)
k = 5 * params.n_ppairs / 4.0
JL = J @ L_stat
A = k * (J @ L_stat + L_stat @ J.T)

df = load_aggregated_csv_data(AGGREGATED_FILE_PATH, COLUMN_MAP)
X = df[["omega", "id1", "iq1", "id3", "iq3"]].to_numpy(dtype=np.float32)
volt = df[["ud1", "uq1", "ud3", "uq3"]].to_numpy(dtype=np.float32)
torq = df[["torq"]].to_numpy(dtype=np.float32)

X_trainval, X_test, volt_trainval, volt_test, torq_trainval, torq_test = train_test_split(
    X, volt, torq, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_train, X_val, volt_train, volt_val, torq_train, torq_val = train_test_split(
    X_trainval, volt_trainval, torq_trainval, test_size=VAL_SIZE, random_state=RANDOM_STATE
)
print(f"train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")
print(f"Using compute device: {DEVICE}")

Loaded 174 valid data points from CSV.
train=117  val=30  test=27
Using compute device: cpu


In [3]:
import math


def _gelu_deriv(z):
    return 0.5 * (1.0 + torch.erf(z / math.sqrt(2.0))) + z * torch.exp(-0.5 * z * z) / math.sqrt(2.0 * math.pi)


def _gelu_second_deriv(z):
    return (2.0 - z * z) * torch.exp(-0.5 * z * z) / math.sqrt(2.0 * math.pi)


def batched_grad_hessian(model, x_normed, scaler_scale):
    """Analytic per-sample gradient and Hessian of the network's scalar
    output w.r.t. current, matching NeuralFlux's numpy inference formula
    exactly -- a forward-mode 2nd-order propagation through model.layers
    (an arbitrary number of GELU hidden layers) followed by the linear
    model.out_layer, generalizing the old single-hidden-layer closed form
    (W1.T @ diag(w2*act''(z)) @ W1) to any depth. Symmetric by construction
    at every depth (Hessian of a genuine scalar function), verified against
    finite differences and the old depth=1 formula to ~1e-8 before use here.
    Ordinary tensor ops, differentiable w.r.t. the weights like any other
    forward pass, so normal backprop trains through this correctly -- no
    autograd.grad/double backprop needed.

    Returns (grad_i, hess_i): d(W_res)/di, d^2(W_res)/di^2, shapes (N, dim)
    and (N, dim, dim) -- already restricted to current dims (omega dropped)
    and divided through by the scaler scale (chain rule for x_normed).
    """
    N, d0 = x_normed.shape
    scale = torch.as_tensor(scaler_scale, dtype=torch.float32)

    grad_h = torch.eye(d0, dtype=torch.float32).unsqueeze(0).expand(N, d0, d0)  # (N, n_prev=d0, d0)
    hess_h = torch.zeros(N, d0, d0, d0, dtype=torch.float32)  # (N, n_prev=d0, d0, d0)
    h = x_normed

    for layer in model.layers:
        W, b = layer.weight, layer.bias  # (n_k, n_prev), (n_k,)
        z = h @ W.T + b  # (N, n_k)
        grad_z = torch.einsum("kp,npd->nkd", W, grad_h)  # (N, n_k, d0)
        hess_z = torch.einsum("kp,npqr->nkqr", W, hess_h)  # (N, n_k, d0, d0)

        d_act = _gelu_deriv(z)
        d2_act = _gelu_second_deriv(z)

        h = torch.nn.functional.gelu(z)
        grad_h = d_act.unsqueeze(-1) * grad_z  # (N, n_k, d0)
        hess_h = d_act.unsqueeze(-1).unsqueeze(-1) * hess_z + d2_act.unsqueeze(-1).unsqueeze(-1) * torch.einsum(
            "nkp,nkq->nkpq", grad_z, grad_z
        )

    w_out = model.out_layer.weight.reshape(-1)  # (n_last,) -- output_size == 1
    grad_normed = torch.einsum("k,nkd->nd", w_out, grad_h)  # (N, 5)
    hess_normed = torch.einsum("k,nkpq->npq", w_out, hess_h)  # (N, 5, 5), symmetric per sample

    grad_x = grad_normed / scale
    grad_i = grad_x[:, 1:]  # (N, dim)

    hess_x = hess_normed / (scale.view(1, -1, 1) * scale.view(1, 1, -1))
    hess_i = hess_x[:, 1:, 1:]  # (N, dim, dim)
    return grad_i, hess_i


def physics_loss(model, x_normed, i, omega, volt_meas, torq_meas, var_v, var_t):
    """Voltage/torque loss with flux = flux_pm + L_stat@i + grad(W_res),
    inductance = L_stat + hess(W_res), matching NeuralFlux/PMSMDrive exactly.
    A_batch here MUST match PMSMDrive.torque()'s formula (J@L + L@J.T, NOT
    (J@L)^T -- only equal when L is symmetric) -- the earlier version of
    this notebook had exactly this bug, silently degrading real test
    performance behind an artificially good training-time self-evaluation.

    The combined loss applies VOLT_WEIGHT to the (variance-normalized)
    voltage term -- see its definition above for why (voltage is a hard
    optimizer constraint, torque merely the objective).
    """
    grad_i, hess_i = batched_grad_hessian(model, x_normed, SCALER_SCALE)
    flux_pred = flux_pm.unsqueeze(0) + i @ L_stat.T + grad_i  # (N, dim)
    L_batch = L_stat.unsqueeze(0) + hess_i  # (N, dim, dim)

    JL_batch = torch.einsum("jl,nlk->njk", J, L_batch)
    LJt_batch = torch.einsum("npq,rq->npr", L_batch, J)
    v_pred = (
        i @ R_stat.T
        + omega * torch.einsum("njk,nk->nj", JL_batch, i)
        + omega * torch.einsum("jl,nl->nj", J, flux_pred)
    )
    loss_v = nn.functional.mse_loss(v_pred, volt_meas)

    A_batch = k * (JL_batch + LJt_batch)
    quad = torch.einsum("ni,nij,nj->n", i, A_batch, i).unsqueeze(1)
    b_pred = k * torch.einsum("jl,nl->nj", J, flux_pred)
    t_pred = quad + 2.0 * (b_pred * i).sum(dim=1, keepdim=True)
    loss_t = nn.functional.mse_loss(t_pred, torq_meas)

    return VOLT_WEIGHT * loss_v / var_v + loss_t / var_t, loss_v, loss_t

In [4]:
def to_t(a):
    return torch.from_numpy(np.asarray(a, dtype=np.float32)).to(DEVICE)


scaler_X = StandardScaler()
X_train_norm = to_t(scaler_X.fit_transform(X_train))
X_val_norm = to_t(scaler_X.transform(X_val))
X_test_norm = to_t(scaler_X.transform(X_test))
SCALER_SCALE = scaler_X.scale_

i_train_t, i_val_t, i_test_t = to_t(X_train[:, 1:]), to_t(X_val[:, 1:]), to_t(X_test[:, 1:])
omega_train_t, omega_val_t, omega_test_t = to_t(X_train[:, :1]), to_t(X_val[:, :1]), to_t(X_test[:, :1])
volt_train_t, volt_val_t, volt_test_t = to_t(volt_train), to_t(volt_val), to_t(volt_test)
torq_train_t, torq_val_t, torq_test_t = to_t(torq_train), to_t(torq_val), to_t(torq_test)

var_v = volt_train_t.var()
var_t = torq_train_t.var()

model = NeuralFluxPredictor(INPUT_SIZE, HIDDEN_SIZES, OUTPUT_SIZE, scaler_X, DEVICE, ACTIVATION).to(DEVICE)
with torch.no_grad():
    model.out_layer.weight.mul_(1e-2)
    model.out_layer.bias.mul_(1e-2)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_val_loss = float("inf")
best_weights = copy.deepcopy(model.state_dict())
epochs_no_improve = 0
stopped_epoch = EPOCHS

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    loss, _, _ = physics_loss(model, X_train_norm, i_train_t, omega_train_t, volt_train_t, torq_train_t, var_v, var_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss, val_lv, val_lt = physics_loss(
            model, X_val_norm, i_val_t, omega_val_t, volt_val_t, torq_val_t, var_v, var_t
        )
    val_loss_val = val_loss.item()

    if val_loss_val < best_val_loss - 1e-6:
        best_val_loss = val_loss_val
        best_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            stopped_epoch = epoch + 1
            break

    if epoch % 1000 == 0:
        print(
            f"  epoch {epoch:5d}  train={loss.item():.4f}  val={val_loss_val:.4f}"
            f"  val_rmse_v={val_lv.item()**0.5:.4f}  val_rmse_t={val_lt.item()**0.5:.4f}"
        )

model.load_state_dict(best_weights)
print(f"\nStopped at epoch {stopped_epoch}, val_loss={best_val_loss:.4f}")

model.eval()
with torch.no_grad():
    _, test_lv, test_lt = physics_loss(
        model, X_test_norm, i_test_t, omega_test_t, volt_test_t, torq_test_t, var_v, var_t
    )
print(f"\n[training-formula eval] Test voltage RMSE [V]: {test_lv.item()**0.5:.4f}")
print(f"[training-formula eval] Test torque  RMSE [Nm]: {test_lt.item()**0.5:.4f}")

  epoch     0  train=0.8607  val=1.4546  val_rmse_v=2.2384  val_rmse_t=1.1436


  epoch  1000  train=0.1488  val=0.2209  val_rmse_v=0.8203  val_rmse_t=0.5357


  epoch  2000  train=0.1334  val=0.1827  val_rmse_v=0.7131  val_rmse_t=0.5345


  epoch  3000  train=0.1293  val=0.1643  val_rmse_v=0.6573  val_rmse_t=0.5315



Stopped at epoch 3932, val_loss=0.1596

[training-formula eval] Test voltage RMSE [V]: 0.6143
[training-formula eval] Test torque  RMSE [Nm]: 0.5136


In [5]:
torch.save(model.state_dict(), MODEL_SAVE_PATH)
np.save(SCALER_SAVE_PATH, {"mean": scaler_X.mean_, "scale": scaler_X.scale_})
print(f"Saved weights to {MODEL_SAVE_PATH}")
print(f"Saved scaler to {SCALER_SAVE_PATH}")

Saved weights to ../weights/FluxNN_Weights.pth
Saved scaler to ../weights/FluxNN_Scaler.npy


In [6]:
cpu = torch.device("cpu")
net, loaded_scaler = load_neural_flux_model(
    MODEL_SAVE_PATH, SCALER_SAVE_PATH,
    hidden_sizes=HIDDEN_SIZES, input_size=INPUT_SIZE, output_size=OUTPUT_SIZE, device=cpu,
    activation=ACTIVATION,
)
neural_flux = NeuralFlux(net, loaded_scaler, cpu, params.L_stat, _build_cross_coupling(2), params.flux_pm)
neural_flux_machine = PMSMDrive(params, flux=neural_flux)

v_pred_lib, t_pred_lib = [], []
for row_idx in range(len(X_test)):
    w = float(X_test[row_idx, 0])
    i_row = X_test[row_idx, 1:]
    t_pred_lib.append(neural_flux_machine.torque(w, i_row))
    v_pred_lib.append(neural_flux_machine.voltage_operator(w, i_row) @ i_row + neural_flux_machine.bemf_dq(w, i_row))
v_pred_lib, t_pred_lib = np.array(v_pred_lib), np.array(t_pred_lib)
rmse_v_lib = np.sqrt(np.mean((volt_test - v_pred_lib) ** 2))
rmse_t_lib = np.sqrt(np.mean((torq_test[:, 0] - t_pred_lib) ** 2))
print(f"[library eval, must match training-formula eval above] Test voltage RMSE [V]: {rmse_v_lib:.4f}")
print(f"[library eval, must match training-formula eval above] Test torque  RMSE [Nm]: {rmse_t_lib:.4f}")

print("\n--- Symmetry check (must hold everywhere, by construction) ---")
rng = np.random.default_rng(0)
max_asym = 0.0
for _ in range(200):
    i_rand = rng.uniform(-30, 30, size=4)
    w_rand = rng.uniform(0, 1800)
    L = neural_flux.inductance(w_rand, i_rand)
    max_asym = max(max_asym, np.max(np.abs(L - L.T)))
print(f"max |L - L.T| over 200 random points: {max_asym:.2e} (should be ~0, i.e. machine precision)")

print("\n--- Extrapolation check: the point that gave ~13 Nm with the old architecture ---")
i_bad = np.array([-8.29179607, -25.51952425, -21.70820393, 15.77193336])
t_old_bug = neural_flux_machine.torque(0.0, i_bad)
t_baseline_only = ieee_machine2().torque(0.0, i_bad)
print(f"torque at this point (new co-energy-residual model): {t_old_bug:.4f} Nm")
print(f"torque at this point (analytic ConstantFlux baseline alone): {t_baseline_only:.4f} Nm")
print("(new model should be reasonably close to the baseline here, not a wild multiple of it)")

[library eval, must match training-formula eval above] Test voltage RMSE [V]: 0.6143
[library eval, must match training-formula eval above] Test torque  RMSE [Nm]: 0.5136

--- Symmetry check (must hold everywhere, by construction) ---
max |L - L.T| over 200 random points: 0.00e+00 (should be ~0, i.e. machine precision)

--- Extrapolation check: the point that gave ~13 Nm with the old architecture ---
torque at this point (new co-energy-residual model): -5.2135 Nm
torque at this point (analytic ConstantFlux baseline alone): -5.6235 Nm
(new model should be reasonably close to the baseline here, not a wild multiple of it)


In [7]:
baseline = ieee_machine2()  # ConstantFlux: single joint-least-squares flux_pm, fixed L_stat


def eval_on_test(mach):
    v_pred, t_pred = [], []
    for row_idx in range(len(X_test)):
        w = float(X_test[row_idx, 0])
        i_row = X_test[row_idx, 1:]
        t_pred.append(mach.torque(w, i_row))
        v_pred.append(mach.voltage_operator(w, i_row) @ i_row + mach.bemf_dq(w, i_row))
    v_pred, t_pred = np.array(v_pred), np.array(t_pred)
    rmse_v = np.sqrt(np.mean((volt_test - v_pred) ** 2))
    rmse_t = np.sqrt(np.mean((torq_test[:, 0] - t_pred) ** 2))
    return rmse_v, rmse_t


rmse_v_const, rmse_t_const = eval_on_test(baseline)
rmse_v_neural, rmse_t_neural = eval_on_test(neural_flux_machine)

print(f"n_test = {len(X_test)}\n")
print(f"{'model':<32} {'voltage RMSE [V]':>18} {'torque RMSE [Nm]':>18}")
print(f"{'ConstantFlux (baseline)':<32} {rmse_v_const:>18.4f} {rmse_t_const:>18.4f}")
print(f"{'NeuralFlux (co-energy residual)':<32} {rmse_v_neural:>18.4f} {rmse_t_neural:>18.4f}")

n_test = 27

model                              voltage RMSE [V]   torque RMSE [Nm]
ConstantFlux (baseline)                      0.5502             0.8655
NeuralFlux (co-energy residual)              0.6143             0.5136
